In [3]:
%reload_ext autoreload
%autoreload 2

import pandas as pd
import os
import numpy as np

# import plotly.graph_objects as go
# import plotly.express as px
# from plotly.subplots import make_subplots
# import plotly.io as pio
# pio.renderers.default = 'notebook'
# pio.templates.default = 'simple_white'

import random
# import nltk
# from nltk.tokenize import word_tokenize
from tqdm import tqdm


import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings(action='ignore', category=UserWarning, module='nltk')
warnings.filterwarnings(action='ignore', category=UserWarning, module='tensorflow')

import logging
logging.getLogger('VoskAPI').setLevel(logging.CRITICAL)

In [4]:
import os
import shutil
import tempfile
from subprocess import run, PIPE

In [ ]:
def compute_gop_with_kaldi(
    wav_path: str,
    transcript: str,
    kaldi_root: str,
    model_dir: str,
    extractor_dir: str,
    lang_dir: str,
    conf_dir: str
):
    # 1) make a temp workdir
    wd = tempfile.mkdtemp(prefix="kaldi_gop_")
    data = os.path.join(wd, "data")
    os.makedirs(data)
    # 2) write wav.scp, text, utt2spk, spk2utt
    utt = "utt1"
    with open(f"{data}/wav.scp", "w") as f:
        f.write(f"{utt} {wav_path}\n")
    with open(f"{data}/text", "w") as f:
        f.write(f"{utt} {transcript}\n")
    with open(f"{data}/utt2spk", "w") as f:
        f.write(f"{utt} spk1\n")
    with open(f"{data}/spk2utt", "w") as f:
        f.write("spk1 utt1\n")

    # 3) define output dirs
    mfccdir = os.path.join(wd, "mfcc")
    ivectordir = os.path.join(wd, "ivectors")
    probdir = os.path.join(wd, "probs")
    alidir = os.path.join(wd, "align")
    gopdir = os.path.join(wd, "gop")
    for d in (mfccdir, ivectordir, probdir, alidir, gopdir):
        os.makedirs(d)

    kaldi_s5_dir = "/Users/veronicabossio/kaldi/egs/librispeech/s5"

    # 4) run Kaldi stages
    stages = [
        # high-res MFCC + CMVN
        f"steps/make_mfcc.sh --nj 1 --mfcc-config {conf_dir}/mfcc_hires.conf data test {mfccdir}",
        f"steps/compute_cmvn_stats.sh data test {mfccdir} {mfccdir}",
        # online i-vectors
        f"steps/online/nnet2/extract_ivectors_online.sh --nj 1 data test {extractor_dir} {ivectordir}",
        # frame-posteriors
        f"steps/nnet3/compute_output.sh --nj 1 --online-ivector-dir {ivectordir} data test {model_dir} {probdir}",
        # forced-alignment
        f"steps/nnet3/align.sh --nj 1 --use_gpu false --online_ivector_dir {ivectordir} data test {lang_dir} {model_dir} {alidir}",
        # strip stress markers
        f"local/remove_phone_markers.pl {lang_dir}/phones.txt {alidir}/phones-pure.txt {alidir}/phone-to-pure-phone.int",
        # convert align to phones
        f"ali-to-phones --per-frame=true {model_dir}/final.mdl ark,t:gunzip -c {alidir}/ali.1.gz ark,t:|gzip -c >{alidir}/ali-phone.1.gz",
        # compute GOP
        f"compute-gop --phone-map={alidir}/phone-to-pure-phone.int {model_dir}/final.mdl ark,t:gunzip -c {alidir}/ali-phone.1.gz ark:{probdir}/output.1.ark ark,t:{gopdir}/gop.1.txt ark,t:{gopdir}/phonefeat.1.txt"
    ]

    for cmd in stages:
        print(f">>> {cmd}")
        run(cmd, shell=True, cwd=kaldi_root, check=True)
        run(cmd, shell=True, check=True, cwd=kaldi_s5_dir)


    # 5) read back the GOP text
    with open(f"{gopdir}/gop.1.txt") as f:
        lines = f.readlines()
    # clean up if you like:
    # shutil.rmtree(wd)
    return lines

In [6]:
kaldi_root    = os.environ["CONDA_PREFIX"]
model_dir     = "/Users/veronicabossio/kaldi_models/librispeech/exp/chain_cleaned/tdnn1f_2048_sp_bi"
extractor_dir = "/Users/veronicabossio/kaldi_models/librispeech/exp/nnet3_cleaned/extractor"
lang_dir      = "/Users/veronicabossio/kaldi_models/librispeech/data/lang_test_tgsmall"
conf_dir      = "/Users/veronicabossio/kaldi_models/librispeech/exp/chain_cleaned/tdnn1f_2048_sp_bi/conf"

wav_path      = "/Users/veronicabossio/Library/Mobile Documents/com~apple~CloudDocs/brooklyn_health/data/MACHINE.wav"
transcript    = "MACHINE"

In [8]:
%ls kaldi_root

ls: kaldi_root: No such file or directory


In [7]:
gop_output = compute_gop_with_kaldi(
    wav_path, transcript,
    kaldi_root, model_dir, extractor_dir, lang_dir, conf_dir
)
print("GOP result:", gop_output)

>>> steps/make_mfcc.sh --nj 1 --mfcc-config /Users/veronicabossio/kaldi_models/librispeech/exp/chain_cleaned/tdnn1f_2048_sp_bi/conf/mfcc_hires.conf data test /var/folders/q6/6x90bffd3vd1s2tm251m8v2m0000gn/T/kaldi_gop_9raj0lkh/mfcc


/bin/sh: steps/make_mfcc.sh: No such file or directory


CalledProcessError: Command 'steps/make_mfcc.sh --nj 1 --mfcc-config /Users/veronicabossio/kaldi_models/librispeech/exp/chain_cleaned/tdnn1f_2048_sp_bi/conf/mfcc_hires.conf data test /var/folders/q6/6x90bffd3vd1s2tm251m8v2m0000gn/T/kaldi_gop_9raj0lkh/mfcc' returned non-zero exit status 127.